## Table of contents

This modelling pipeline is subdivided in the following sections:

3. [Dimensionality reduction](#3)
   1. [Low-dimensional representation](#31)
   2. [Clustering](#32)
   3. [Analysis](#33)



![State of the workflow](images/DimensionalityReduction.png)

To get started, let's load some packages!

In [ ]:
# File and system operations
import os
import sys
import subprocess
from glob import glob
import pickle
import shutil

# Data processing
import pandas as pd
import numpy as np
import mdtraj as md

# Network and parallel processing
import requests
import time
import multiprocessing
import concurrent.futures

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns

# custom utility functions and class
from workflow.utilities import (
    count_pdb_files,
    braf_res,
    clear_and_make,
    make_seg,
    copy_filtered_pdbs,
    copy_cg_chain_small_molecules,
)
from workflow.utilities import PDBDownloader

# 3. Dimensionality reduction  <a id="3"></a>
In this section we will apply dimensionality reduction to our coarse-grained representation of selected activation loops.

## 2.6 Coarse-graining activation loops <a id="26"></a>
Here we coarse-grain activation loops to a uniform number of backbone points, independently of the original loop length. Full chains in `Results/activation_segments/misaligned_filter/` are CA-stripped on the fly (DFG→APE) and resampled to 27 points.


### 2.5.2 Cα interpolation / coarse-graining


We prepare a uniform input for dimensionality reduction. Loops have different numbers of residues, so each DFG→APE segment is either copied (if it already has 27 Cαs) or cubic-spline fitted and resampled to 27 equidistant points.


We use `Fitting(n_ca_template=27, copy_exact_length=True)` with `process_chains_directory`: it strips the activation loop from each full chain, copies exact-length 27-CA loops unchanged onto the template, and spline-fits all others. Output is written to `Results/activation_segments/fitted/`.


In [ ]:
from workflow.fitting_class import Fitting

CHAINS_DIR = "Results/activation_segments/misaligned_filter/"
FITTED_DIR = "Results/activation_segments/fitted"
N_CG = 27  # number of equidistant backbone points for coarse-graining

# CA-strip the activation loop on-the-fly; 27-CA loops are copied, others are spline-fitted to N_CG points
fitter = Fitting(n_ca_template=N_CG, copy_exact_length=True)
fitter.process_chains_directory(
    input_dir=CHAINS_DIR,
    output_dir=FITTED_DIR,
    motifs=["DFG", "APE"],
)


We also save a copy of the protein–small-molecule complexes for chains in the coarse-grained (fitted) set. Complexes are taken from `Results/motif_filtered_small_molecules/` and written to `Results/CG_chain_small_molecules/`, matched by `PDBID_CHAIN`.


In [ ]:
from workflow.utilities import copy_cg_chain_small_molecules

S_MOLECULES_SRC = "Results/motif_filtered_small_molecules/"
S_MOLECULES_DST = "Results/CG_chain_small_molecules/"
FITTED_DIR = "Results/activation_segments/fitted"

copy_cg_chain_small_molecules(
    small_molecules_src=S_MOLECULES_SRC,
    small_molecules_dst=S_MOLECULES_DST,
    cg_dir=FITTED_DIR,
)
